# EV Range Prediction from Vehicle Specifications

**TECHTRACK 3.0 — EV-Focused ML Case Battle**

### Objective
Build a reproducible regression system that predicts an electric vehicle's driving range (`range_km`) from static vehicle specifications.

This notebook covers data quality analysis, cleaning, EDA, feature selection, preprocessing, model development, hyperparameter tuning, evaluation, error analysis, explainability, ablation testing, and final model persistence.

## 1. Problem Definition

The target is **`range_km`**, the official EV driving range in kilometres. The model uses only static vehicle specifications available in the dataset.

**Scope limitation:** the dataset does not contain dynamic factors such as traffic, weather, HVAC usage, driving style, battery degradation, state of charge, or state of health.

## 2. Imports and Reproducibility

A fixed random seed of **42** is used wherever applicable so that data splitting and randomized searches are reproducible.

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 3. Load Dataset

The final notebook assumes `EV.csv` is available in the same working directory. The file uses `cp1252` encoding because the source contains the degree symbol (`°`).

In [ ]:
DATA_PATH = "EV.csv"
df = pd.read_csv(DATA_PATH, encoding="cp1252")
print("Dataset shape:", df.shape)
df.head()

## 4. Initial Data Audit

Inspect dimensions, dtypes, missing values, and cardinality before transformation.

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nExact duplicate rows:", df.duplicated().sum())

In [ ]:
print("Unique values per column:")
print(df.nunique().sort_values())

## 5. Cleaning and Data Quality

A copy of the raw dataframe is used for analysis. Numeric columns are coerced with `errors="coerce"`, converting invalid numeric text to missing values instead of inventing a value. This addresses the non-numeric `cargo_volume_l` entries such as **"Banana Boxes"**.

In [ ]:
df_clean = df.copy()

numeric_cols = [
    "top_speed_kmh", "battery_capacity_kWh", "number_of_cells",
    "torque_nm", "efficiency_wh_per_km", "range_km",
    "acceleration_0_100_s", "fast_charging_power_kw_dc",
    "towing_capacity_kg", "cargo_volume_l", "seats",
    "length_mm", "width_mm", "height_mm"
]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Shape after cleaning/coercion:", df_clean.shape)
print("\nMissing values after numeric coercion:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

In [ ]:
print("Exact duplicate rows:", df_clean.duplicated().sum())

### Missing-value profile

Main missing-value concentrations:

- `number_of_cells`: 202 (~42.3%)
- `towing_capacity_kg`: 26 (~5.4%)
- `torque_nm`: 7 (~1.5%)
- `cargo_volume_l`: 4 after coercion
- isolated single missing values in `model`, `fast_charge_port`, and `fast_charging_power_kw_dc`

Missing values are handled inside the model pipeline so that imputation is learned only from training folds.

In [ ]:
missing = df_clean.isnull().sum()
missing_percent = (missing / len(df_clean)) * 100
missing_table = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent})
missing_table[missing_table["Missing Count"] > 0].sort_values("Missing %", ascending=False)

## 6. Categorical Feature Audit

`model` is almost unique for every row and is treated as high-cardinality. `battery_type` has zero variance. `source_url` is metadata rather than a predictive vehicle specification.

In [ ]:
categorical_cols = ["brand", "model", "battery_type", "fast_charge_port", "drivetrain", "segment", "car_body_type"]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", df_clean[col].nunique(dropna=True))
    print(df_clean[col].value_counts(dropna=False).head(15))

## 7. Numerical Summary and Outlier Review

Extreme values were reviewed rather than automatically removed. Some apparently extreme values represent legitimate vehicle specifications.

In [ ]:
df_clean[numeric_cols].describe().T

In [ ]:
df_clean.nlargest(10, "acceleration_0_100_s")[["brand", "model", "acceleration_0_100_s", "battery_capacity_kWh", "range_km"]]

In [ ]:
df_clean[df_clean["number_of_cells"] > 1000][["brand", "model", "battery_capacity_kWh", "number_of_cells", "range_km"]].sort_values("number_of_cells", ascending=False)

## 8. Exploratory Data Analysis

### Numerical relationships with range

Correlation is used to understand relationships and support feature decisions; it is not interpreted as causation.

In [ ]:
corr = df_clean[numeric_cols].corr()
corr["range_km"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 8), dpi=150)
sns.heatmap(df_clean[numeric_cols].drop(columns="range_km").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numerical Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Key EDA observations

- `battery_capacity_kWh` has the strongest positive correlation with range (~0.88).
- `top_speed_kmh`, `fast_charging_power_kw_dc`, and `torque_nm` also show positive associations.
- `acceleration_0_100_s` shows a strong negative association with range (~−0.71).
- `length_mm` and `width_mm` are strongly correlated (~0.85).
- `top_speed_kmh` is strongly related to acceleration and torque.

These findings support evaluating both linear and nonlinear regression models.

In [ ]:
feature_corr = df_clean[numeric_cols].drop(columns="range_km").corr().abs()
upper = feature_corr.where(np.triu(np.ones(feature_corr.shape), k=1).astype(bool))
high_corr_pairs = upper.stack().sort_values(ascending=False)
print(high_corr_pairs[high_corr_pairs > 0.80])

### Categorical relationships with range

In [ ]:
print("Drivetrain:")
print(df_clean.groupby("drivetrain")["range_km"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False))

print("\nBody type:")
print(df_clean.groupby("car_body_type")["range_km"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False))

print("\nSegment:")
print(df_clean.groupby("segment")["range_km"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False))

## 9. Leakage Prevention and Feature Selection

### Final 17 model inputs

**Numerical:** `top_speed_kmh`, `battery_capacity_kWh`, `number_of_cells`, `torque_nm`, `acceleration_0_100_s`, `fast_charging_power_kw_dc`, `towing_capacity_kg`, `cargo_volume_l`, `seats`, `length_mm`, `width_mm`, `height_mm`

**Categorical:** `brand`, `fast_charge_port`, `drivetrain`, `segment`, `car_body_type`

### Excluded
- `range_km` — target
- `model` — high cardinality
- `battery_type` — zero variance
- `source_url` — metadata
- **`efficiency_wh_per_km` — excluded from final model inputs to prevent leakage**

In [ ]:
feature_cols = [
    "brand", "top_speed_kmh", "battery_capacity_kWh", "number_of_cells",
    "torque_nm", "acceleration_0_100_s", "fast_charging_power_kw_dc",
    "fast_charge_port", "towing_capacity_kg", "cargo_volume_l", "seats",
    "drivetrain", "segment", "length_mm", "width_mm", "height_mm",
    "car_body_type"
]

target_col = "range_km"
X = df_clean[feature_cols]
y = df_clean[target_col]
print("X shape:", X.shape)
print("y shape:", y.shape)

## 10. Train/Test Split

An 80/20 split is used. The test set remains untouched during model selection and hyperparameter tuning.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 11. Preprocessing Pipeline

Numerical missing values use median imputation. Categorical missing values use most-frequent imputation followed by one-hot encoding. `handle_unknown="ignore"` makes the pipeline robust to unseen categories during prediction.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "top_speed_kmh", "battery_capacity_kWh", "number_of_cells", "torque_nm",
    "acceleration_0_100_s", "fast_charging_power_kw_dc", "towing_capacity_kg",
    "cargo_volume_l", "seats", "length_mm", "width_mm", "height_mm"
]

categorical_features = [
    "brand", "fast_charge_port", "drivetrain", "segment", "car_body_type"
]

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 12. Cross-Validation Setup

Five-fold shuffled K-Fold cross-validation is used for model selection. MAE is the main tuning metric because it is directly interpretable in kilometres.

In [ ]:
from sklearn.model_selection import KFold, cross_validate

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 13. Baseline — Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

cv_results = cross_validate(
    linear_model, X_train, y_train, cv=kf,
    scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"},
    return_train_score=False
)

mae_scores = -cv_results["test_MAE"]
rmse_scores = -cv_results["test_RMSE"]
r2_scores = cv_results["test_R2"]

print("5-Fold CV Results")
print("MAE :", mae_scores.mean())
print("RMSE:", rmse_scores.mean())
print("R²  :", r2_scores.mean())
print("\nStandard deviations")
print("MAE :", mae_scores.std())
print("RMSE:", rmse_scores.std())
print("R²  :", r2_scores.std())

## 14. Model Development and Hyperparameter Tuning

Nine regression families were evaluated. The tuning code below reproduces the searches used during development; the final summary records the completed search results.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform

### Ridge — best result
Best α = **1**; best CV MAE = **14.988 km**.

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Ridge())
])
param_grid_ridge = {"model__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000, 5000, 10000]}
grid_ridge = GridSearchCV(ridge_model, param_grid_ridge, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_ridge.fit(X_train, y_train)
print("Best Parameters:", grid_ridge.best_params_)
print("Best CV MAE:", -grid_ridge.best_score_)

### Lasso — best result
Best α = **0.1**; best CV MAE = **14.873 km**.

In [ ]:
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Lasso(max_iter=10000))
])
param_grid_lasso = {"model__alpha": [0.0001, 0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100]}
grid_lasso = GridSearchCV(lasso_model, param_grid_lasso, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_lasso.fit(X_train, y_train)
print("Best Parameters:", grid_lasso.best_params_)
print("Best CV MAE:", -grid_lasso.best_score_)

### Elastic Net — best result
Best α = **0.1**, `l1_ratio` = **0.99**; best CV MAE = **14.841 km**.

In [ ]:
elastic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", ElasticNet(max_iter=10000))
])
param_grid_elastic = {
    "model__alpha": [0.1, 1, 5, 10, 50, 100],
    "model__l1_ratio": [0.1, 0.5, 0.7, 0.95, 0.99, 1]
}
grid_elastic = GridSearchCV(elastic_model, param_grid_elastic, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_elastic.fit(X_train, y_train)
print("Best Parameters:", grid_elastic.best_params_)
print("Best CV MAE:", -grid_elastic.best_score_)

### SVR — best result
The broader SVR search found **C = 10000**, **epsilon = 2**, **gamma = 0.001**, with **CV MAE = 12.004 km**.

In [ ]:
from sklearn.svm import SVR

svr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", SVR())
])

param_grid_svr = {
    "model__C": [500, 1000, 2000, 5000, 10000],
    "model__epsilon": [0.01, 0.1, 0.5, 1, 2, 5],
    "model__gamma": ["scale", "auto", 0.0001, 0.0005, 0.001, 0.005, 0.01]
}

grid_search_svr = GridSearchCV(svr_model, param_grid_svr, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_search_svr.fit(X_train, y_train)

print("Best Parameters:", grid_search_svr.best_params_)
print("Best CV MAE:", -grid_search_svr.best_score_)

### Decision Tree — best result
Best CV MAE = **19.520 km** after fine tuning.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))
])
param_grid_tree = {
    "model__max_depth": [3, 4, 5, 6, 7, 8, 9, 10, 12, 15],
    "model__min_samples_split": [2, 3, 5, 8, 10, 15, 20],
    "model__min_samples_leaf": [1, 2, 3, 5, 8, 10, 15],
    "model__max_features": [None, "sqrt", "log2"]
}
grid_tree = GridSearchCV(tree_model, param_grid_tree, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_tree.fit(X_train, y_train)
print("Best Parameters:", grid_tree.best_params_)
print("Best CV MAE:", -grid_tree.best_score_)

### Random Forest — best result
Randomized tuning produced a best CV MAE of **16.384 km**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])
param_dist_rf = {
    "model__n_estimators": randint(100, 500),
    "model__max_depth": [None, 5, 8, 10, 12, 15, 20, 25],
    "model__min_samples_split": randint(2, 15),
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": [1.0, "sqrt", "log2", 0.5, 0.7],
    "model__min_impurity_decrease": [0.0, 0.00001, 0.0001, 0.001, 0.01]
}
random_rf = RandomizedSearchCV(rf_model, param_dist_rf, n_iter=50, cv=5, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
random_rf.fit(X_train, y_train)
print("Best Parameters:", random_rf.best_params_)
print("Best CV MAE:", -random_rf.best_score_)

### Gradient Boosting — best result
Randomized tuning produced a best CV MAE of **13.085 km**.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))
])
param_dist_gb = {
    "model__n_estimators": randint(100, 500),
    "model__learning_rate": uniform(0.01, 0.19),
    "model__max_depth": randint(2, 10),
    "model__min_samples_split": randint(2, 15),
    "model__min_samples_leaf": randint(1, 10),
    "model__subsample": uniform(0.6, 0.4),
    "model__max_features": [None, "sqrt", "log2"]
}
random_gb = RandomizedSearchCV(gb_model, param_dist_gb, n_iter=50, cv=5, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
random_gb.fit(X_train, y_train)
print("Best Parameters:", random_gb.best_params_)
print("Best CV MAE:", -random_gb.best_score_)

### XGBoost — best result
Randomized tuning produced a best CV MAE of **12.492 km**.

In [ ]:
from xgboost import XGBRegressor

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1))
])
param_dist_xgb = {
    "model__n_estimators": randint(100, 600),
    "model__learning_rate": uniform(0.01, 0.19),
    "model__max_depth": randint(2, 10),
    "model__min_child_weight": randint(1, 10),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__gamma": uniform(0, 0.5),
    "model__reg_alpha": uniform(0, 1),
    "model__reg_lambda": uniform(0.5, 2)
}
random_xgb = RandomizedSearchCV(xgb_model, param_dist_xgb, n_iter=50, cv=5, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
random_xgb.fit(X_train, y_train)
print("Best Parameters:", random_xgb.best_params_)
print("Best CV MAE:", -random_xgb.best_score_)

## 15. Cross-Validation Model Summary

The following values are the recorded best CV MAE values from the completed runs.

In [ ]:
cv_summary = pd.DataFrame({
    "Model": ["SVR", "XGBoost", "Gradient Boosting", "Linear Regression", "Elastic Net", "Lasso", "Ridge", "Random Forest", "Decision Tree"],
    "Best CV MAE (km)": [12.003563, 12.491633, 13.085183, 14.645878, 14.840784, 14.873313, 14.987968, 16.383501, 19.519642]
}).sort_values("Best CV MAE (km)")
cv_summary

## 16. Final Hold-out Test Evaluation

Only after model selection do we evaluate the leading candidates on the untouched test set.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_svr = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", SVR(C=10000, epsilon=2, gamma=0.001))
])
best_svr.fit(X_train, y_train)

best_xgb = random_xgb.best_estimator_
best_gb = random_gb.best_estimator_

y_pred_svr = best_svr.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)
y_pred_gb = best_gb.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R²": r2_score(y_true, y_pred)
    }

final_test_results = pd.DataFrame([
    evaluate_model("SVR", y_test, y_pred_svr),
    evaluate_model("XGBoost", y_test, y_pred_xgb),
    evaluate_model("Gradient Boosting", y_test, y_pred_gb)
]).sort_values("MAE")

final_test_results

### Final model selection rationale

| Model | MAE (km) | RMSE (km) | R² |
|---|---:|---:|---:|
| Gradient Boosting | **9.966** | 13.822 | 0.9819 |
| **SVR** | 10.309 | **13.646** | **0.9824** |
| XGBoost | 10.353 | 14.481 | 0.9802 |

Gradient Boosting has the lowest MAE by about 0.34 km. SVR has lower RMSE, higher R², and a substantially lower maximum absolute error. Therefore, **SVR is selected as the final model** to balance typical error with control of larger errors.

## 17. Error Analysis

Residual = **Actual − Predicted**. A positive residual means underprediction; a negative residual means overprediction.

In [ ]:
errors_svr = y_test - y_pred_svr
errors_gb = y_test - y_pred_gb

print("SVR mean residual:", errors_svr.mean())
print("Gradient Boosting mean residual:", errors_gb.mean())
print("\nSVR maximum absolute error:", np.max(np.abs(errors_svr)))
print("Gradient Boosting maximum absolute error:", np.max(np.abs(errors_gb)))

In [ ]:
print("SVR")
print("Underestimated:", np.sum(errors_svr > 0))
print("Overestimated:", np.sum(errors_svr < 0))
print("Underestimation %:", np.mean(errors_svr > 0) * 100)
print("Overestimation %:", np.mean(errors_svr < 0) * 100)

print("\nGradient Boosting")
print("Underestimated:", np.sum(errors_gb > 0))
print("Overestimated:", np.sum(errors_gb < 0))
print("Underestimation %:", np.mean(errors_gb > 0) * 100)
print("Overestimation %:", np.mean(errors_gb < 0) * 100)

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_svr)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Actual Range (km)")
plt.ylabel("Predicted Range (km)")
plt.title("SVR — Actual vs Predicted Range")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_pred_svr, errors_svr)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Range (km)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("SVR Residual Plot")
plt.tight_layout()
plt.show()

### Error interpretation

Residuals are generally distributed around zero without a strong visible systematic pattern. The mean residual is slightly negative, indicating a small overall overprediction tendency. The maximum absolute SVR error is about **41.19 km**, versus **56.92 km** for Gradient Boosting.

## 18. Worst Prediction Cases

Reviewing the largest errors helps distinguish difficult but legitimate observations from obvious data-quality problems.

In [ ]:
svr_error_df = X_test.copy()
svr_error_df["actual_range"] = y_test
svr_error_df["predicted_range"] = y_pred_svr
svr_error_df["error"] = errors_svr
svr_error_df["absolute_error"] = np.abs(errors_svr)
svr_error_df.sort_values("absolute_error", ascending=False).head(10)

The largest errors are legitimate EV observations rather than obvious garbage records, so they are not removed simply because the model finds them difficult.

## 19. Model Explainability — Permutation Importance

SVR does not expose tree-style `feature_importances_`. Permutation importance estimates how much predictive performance deteriorates when an input feature is shuffled. **It indicates predictive contribution, not causality.**

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_svr, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
importance_df

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(importance_df["feature"].head(15)[::-1], importance_df["importance"].head(15)[::-1])
plt.xlabel("Permutation Importance")
plt.ylabel("Feature")
plt.title("SVR Feature Importance")
plt.tight_layout()
plt.show()

### Explainability finding

`battery_capacity_kWh` is the dominant predictive feature. Other useful features include height, brand, towing capacity, body type, width, drivetrain, charging power, and segment.

## 20. Ablation Study — `number_of_cells`

Although `number_of_cells` has approximately 42.3% missing values, we tested whether removing it improves performance.

| Configuration | MAE (km) | RMSE (km) | R² |
|---|---:|---:|---:|
| With `number_of_cells` | **10.309** | **13.646** | **0.9824** |
| Without `number_of_cells` | 10.813 | 14.095 | 0.9812 |

Removing the feature degraded all three metrics, so it is retained with median imputation.

In [ ]:
numeric_features_no_cells = [
    "top_speed_kmh", "battery_capacity_kWh", "torque_nm",
    "acceleration_0_100_s", "fast_charging_power_kw_dc",
    "towing_capacity_kg", "cargo_volume_l", "seats",
    "length_mm", "width_mm", "height_mm"
]

preprocessor_no_cells = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features_no_cells),
    ("cat", categorical_transformer, categorical_features)
])

svr_no_cells = Pipeline([
    ("preprocessor", preprocessor_no_cells),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", SVR(C=10000, epsilon=2, gamma=0.001))
])
svr_no_cells.fit(X_train, y_train)
y_pred_no_cells = svr_no_cells.predict(X_test)

print("SVR without number_of_cells")
print("MAE :", mean_absolute_error(y_test, y_pred_no_cells))
print("RMSE:", mean_squared_error(y_test, y_pred_no_cells) ** 0.5)
print("R²  :", r2_score(y_test, y_pred_no_cells))

## 21. Save Final Reproducible Model

The saved artifact contains preprocessing and SVR together, allowing the deployment app to pass the raw 17-feature dataframe directly.

In [ ]:
import joblib

MODEL_PATH = "ev_range_svr_pipeline.joblib"
joblib.dump(best_svr, MODEL_PATH)
print("Model saved successfully:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))

## 22. Interactive Deployment

The saved pipeline is used by the Streamlit application for live EV range prediction.

**Live demo:** https://ev-range-prediction-5quiycdpj8khgu6pdba4w3.streamlit.app/

## 23. Final Conclusion

A complete, leakage-aware regression workflow was developed for EV range prediction. Nine regression families were compared and tuned. SVR was selected as the final model because it provides strong typical accuracy while better controlling larger prediction errors on the held-out test set.

**Final SVR test performance:**
- **MAE:** 10.309 km
- **RMSE:** 13.646 km
- **R²:** 0.9824

The model is packaged as a reproducible `.joblib` pipeline and deployed through an interactive Streamlit application.

## 24. Reproducibility Checklist

- Fixed random seed: **42**
- Held-out 20% test set separated from tuning
- Preprocessing contained inside sklearn pipelines
- Categorical unseen values handled with `handle_unknown="ignore"`
- Missing values handled inside the pipeline
- `efficiency_wh_per_km` excluded from final model inputs
- Final model saved as `ev_range_svr_pipeline.joblib`
- Interactive Streamlit application deployed